Banking Customer Query Classification

Objective:
Build a model that classifies customer banking messages into
the appropriate category to support automated query routing.

Input: A customer banking message.
Output: The predicted banking query category.

**Loading the data directly from link...**


https://github.com/PolyAI-LDN/task-specific-datasets/tree/master/banking_data
in above url train.csv and test.csv is alrdy available.So we will use the same train and test csv file...

In [2]:
import pandas as pd

url = (
    "https://raw.githubusercontent.com/"
    "PolyAI-LDN/task-specific-datasets/master/banking_data/"
)

data = pd.read_csv(url + "train.csv")
test_data = pd.read_csv(url + "test.csv")

print("Training data:", data.shape)
print("Test data:" , test_data.shape)
display(data.head())


Training data: (10003, 2)
Test data: (3080, 2)


,text,category
0,I am still waiting on my card?,card_arrival
1,What can I do if my card still hasn't arrived ...,card_arrival
2,I have been waiting over a week. Is the card s...,card_arrival
3,Can I track my card while it is in the process...,card_arrival
4,"How do I know if I will get my card, or if it ...",card_arrival


**Checking the shape which  means total number of records and columns in both train and test data.**

In [3]:
print("Training data:", data.shape)
print("Test data:", test_data.shape)

print("\nColumn names:")
print(data.columns.tolist())

print("\nFirst customer message:")
print(data.iloc[0]["text"])

print("\nIts correct category:")
print(data.iloc[0]["category"])

Training data: (10003, 2)
Test data: (3080, 2)

Column names:
['text', 'category']

First customer message:
I am still waiting on my card?

Its correct category:
card_arrival


**Checking total number of different catergories of messages available...and total number of messages in each category.**

In [4]:
# Training data mein kitni different categories hain?
print("Number of categories:", data["category"].nunique())

# Har category ke kuch messages ki count
print("\nMessages per category:")
print(data["category"].value_counts().head(10))

Number of categories: 77

Messages per category:
category
card_payment_fee_charged                            187
direct_debit_payment_not_recognised                 182
balance_not_updated_after_cheque_or_cash_deposit    181
wrong_amount_of_cash_received                       180
cash_withdrawal_charge                              177
transaction_charged_twice                           175
declined_cash_withdrawal                            173
transfer_fee_charged                                172
balance_not_updated_after_bank_transfer             171
transfer_not_received_by_recipient                  171
Name: count, dtype: int64


Checking for missing value in each column in both train and test both the dataset.
Also the duplicate rows in both dataset and blank or spaces only messages in both dataset.

In [5]:
# 1. Missing values in each column
print("Training missing values:")
print(data.isna().sum())

print("\nTest missing values:")
print(test_data.isna().sum())

# 2. Exact duplicate rows
print("\nTraining duplicates:", data.duplicated().sum())
print("Test duplicates:", test_data.duplicated().sum())

# 3. Blank or spaces-only messages
print("\nTraining empty messages:",
      data["text"].fillna("").str.strip().eq("").sum())

print("Test empty messages:",
      test_data["text"].fillna("").str.strip().eq("").sum())

Training missing values:
text        0
category    0
dtype: int64

Test missing values:
text        0
category    0
dtype: int64

Training duplicates: 0
Test duplicates: 0

Training empty messages: 0
Test empty messages: 0


**Below code is to check if same message is available in more than 1 categories.**

In [6]:
# Temporary comparison key:
# ignore capital letters and extra spaces
text_key = (
    data["text"]
    .str.lower()
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

# Count different categories assigned to each message
category_counts = data.groupby(text_key)["category"].nunique()

conflicts = category_counts[category_counts > 1]

print("Messages with conflicting categories:", len(conflicts))

# Show examples, if any
if len(conflicts) > 0:
    display(data.loc[text_key.isin(conflicts.index)]
            .sort_values("text")
            .head(20))

Messages with conflicting categories: 0


In [7]:
# Normalize text only for comparison
train_keys = (
    data["text"]
    .str.lower()
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

test_keys = (
    test_data["text"]
    .str.lower()
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

# Find messages present in both datasets
overlap = set(train_keys) & set(test_keys)

print("Unique messages shared by train and test:", len(overlap))

Unique messages shared by train and test: 7


In [8]:
print("Matching training rows:")
display(data.loc[train_keys.isin(overlap)])

print("Matching test rows:")
display(test_data.loc[test_keys.isin(overlap)])

Matching training rows:


,text,category
1722,\nHow do I unblock my PIN?,pin_blocked
3103,What businesses accept this card?,card_acceptance
3116,\nWhere can I use my card?,card_acceptance
4476,There are a few transaction that I don't recog...,compromised_card
4576,\nAt which ATMs can I use this card?,atm_support
6984,\nWhich cash machines will allow me to change ...,change_pin
9921,I don't live in the UK. Can I still get a card?,country_support


Matching test rows:


,text,category
554,How do I unblock my PIN?,pin_blocked
976,\n\nWhat businesses accept this card?,card_acceptance
977,Where can I use my card?,card_acceptance
1432,There are a few transaction that I don't recog...,compromised_card
1474,At which ATMs can I use this card?,atm_support
2149,Which cash machines will allow me to change my...,change_pin
3070,I don't live in the UK. Can I still get a card?,country_support


In [9]:
# Keep a separate cleaned training dataset
data_clean = data.loc[~train_keys.isin(overlap)].copy()
data_clean = data_clean.reset_index(drop=True)

print("Original training rows:", len(data))
print("Cleaned training rows:", len(data_clean))
print("Test rows:", len(test_data))

Original training rows: 10003
Cleaned training rows: 9996
Test rows: 3080


In [10]:
category_counts = data_clean["category"].value_counts()

print("Total categories:", len(category_counts))
print("Smallest category count:", category_counts.min())
print("Largest category count:", category_counts.max())

print("\n10 categories with the fewest messages:")
print(category_counts.sort_values().head(10))

Total categories: 77
Smallest category count: 35
Largest category count: 187

10 categories with the fewest messages:
category
contactless_not_working        35
virtual_card_not_working       41
card_acceptance                57
card_swallowed                 61
lost_or_stolen_card            82
compromised_card               85
atm_support                    86
receiving_money                95
top_up_limits                  97
get_disposable_virtual_card    97
Name: count, dtype: int64


In [11]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    data_clean["text"],
    data_clean["category"],
    test_size=0.20,
    random_state=42,
    stratify=data_clean["category"]
)

print("Training messages:", len(X_train))
print("Validation messages:", len(X_val))
print("Training categories:", y_train.nunique())
print("Validation categories:", y_val.nunique())

Training messages: 7996
Validation messages: 2000
Training categories: 77
Validation categories: 77


In [12]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1, 2)
)

# Training text se vocabulary aur scoring rules seekhna
X_train_tfidf = vectorizer.fit_transform(X_train)

# Wahi rules validation text par apply karna
X_val_tfidf = vectorizer.transform(X_val)

print("Training shape:", X_train_tfidf.shape)
print("Validation shape:", X_val_tfidf.shape)

Training shape: (7996, 20915)
Validation shape: (2000, 20915)


**Word to vec Transformation**

In [13]:
%pip -q install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 45.8 MB/s eta 0:00:00


In [14]:
from gensim.utils import tokenize

def get_tokens(message):
    return list(tokenize(message.lower()))

# Convert each message into a list of words
train_tokens = X_train.apply(get_tokens)
val_tokens = X_val.apply(get_tokens)

print("Original message:")
print(X_train.iloc[0])

print("\nTokens:")
print(train_tokens.iloc[0])

Original message:
I can't find my card! Can you help?

Tokens:
['i', 'can', 't', 'find', 'my', 'card', 'can', 'you', 'help']


In [15]:
%pip -q install contractions

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.1/345.1 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.8/114.8 kB 5.2 MB/s eta 0:00:00


In [16]:
import contractions
from gensim.utils import tokenize

def get_tokens(message):
    # Standardize apostrophes and expand contractions
    text = message.lower().replace("’", "'")
    text = contractions.fix(text)

    return list(tokenize(text))

train_tokens = X_train.apply(get_tokens)
val_tokens = X_val.apply(get_tokens)

print("Original message:")
print(X_train.iloc[0])

print("\nUpdated tokens:")
print(train_tokens.iloc[0])

Original message:
I can't find my card! Can you help?

Updated tokens:
['i', 'cannot', 'find', 'my', 'card', 'can', 'you', 'help']


In [17]:
from gensim.models import Word2Vec

w2v_model = Word2Vec(
    sentences=train_tokens.tolist(),
    vector_size=100,
    window=5,
    min_count=1,
    sg=1,
    epochs=20,
    workers=1,
    seed=42
)

print("Vocabulary size:", len(w2v_model.wv))
print("Vector shape for 'card':", w2v_model.wv["card"].shape)
print("First 10 values:", w2v_model.wv["card"][:10])

Vocabulary size: 2122
Vector shape for 'card': (100,)
First 10 values: [ 0.37501642  0.52160364 -0.15398484 -0.12182777  0.14476463  0.33889005
  0.27101433  0.00457417  0.00395455 -0.3300534 ]


In [18]:
import numpy as np

def get_message_vector(tokens):
    # Only include words available in the learned vocabulary
    vectors = [
        w2v_model.wv[word]
        for word in tokens
        if word in w2v_model.wv
    ]

    if len(vectors) > 0:
        return np.mean(vectors, axis=0)

    # If no words are known, return a zero vector
    return np.zeros(w2v_model.vector_size, dtype=np.float32)

# One vector per message
X_train_w2v = np.vstack(train_tokens.apply(get_message_vector))
X_val_w2v = np.vstack(val_tokens.apply(get_message_vector))

print("Training shape:", X_train_w2v.shape)
print("Validation shape:", X_val_w2v.shape)

print("Training has NaN:", np.isnan(X_train_w2v).any())
print("Validation has NaN:", np.isnan(X_val_w2v).any())

Training shape: (7996, 100)
Validation shape: (2000, 100)
Training has NaN: False
Validation has NaN: False


In [19]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()

# Learn category-to-number mapping from training labels
y_train_encoded = label_encoder.fit_transform(y_train)

# Use the same mapping for validation labels
y_val_encoded = label_encoder.transform(y_val)

print("Number of categories:", len(label_encoder.classes_))

print("\nExample mapping:")
for category, number in zip(
    y_train.iloc[:5],
    y_train_encoded[:5]
):
    print(f"{category} → {number}")

Number of categories: 77

Example mapping:
lost_or_stolen_card → 42
wrong_exchange_rate_for_cash_withdrawal → 76
transfer_not_received_by_recipient → 66
card_payment_fee_charged → 16
cancel_transfer → 9


In [20]:
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Input, Dense, Dropout

tf.keras.utils.set_random_seed(42)

model = Sequential([
    Input(shape=(100,)),

    Dense(128, activation="relu"),
    Dropout(0.3),

    Dense(64, activation="relu"),
    Dropout(0.2),

    Dense(77, activation="softmax")
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 128)            │        12,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 77)             │         5,005 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 26,189 (102.30 KB)

 Trainable params: 26,189 (102.30 KB)

 Non-trainable params: 0 (0.00 B)

In [21]:
from tensorflow.keras.callbacks import EarlyStopping

early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

history = model.fit(
    X_train_w2v,
    y_train_encoded,
    validation_data=(X_val_w2v, y_val_encoded),
    epochs=100,
    batch_size=16,
    callbacks=[early_stopping]
)

Epoch 1/100
500/500 ━━━━━━━━━━━━━━━━━━━━ 8s 11ms/step - accuracy: 0.1491 - loss: 3.4947 - val_accuracy: 0.4335 - val_loss: 2.1841
Epoch 2/100
500/500 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - accuracy: 0.3864 - loss: 2.0562 - val_accuracy: 0.6165 - val_loss: 1.4678
Epoch 3/100
500/500 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.5028 - loss: 1.6134 - val_accuracy: 0.6810 - val_loss: 1.1850
Epoch 4/100
500/500 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.5652 - loss: 1.3963 - val_accuracy: 0.7045 - val_loss: 1.0546
Epoch 5/100
500/500 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.6049 - loss: 1.2731 - val_accuracy: 0.7205 - val_loss: 0.9715
Epoch 6/100
500/500 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.6339 - loss: 1.1826 - val_accuracy: 0.7375 - val_loss: 0.9101
Epoch 7/100
500/500 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.6446 - loss: 1.1204 - val_accuracy: 0.7520 - val_loss: 0.8728
Epoch 8/100
500/500 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.6626 - loss: 1.0688 - val_acc

In [22]:
from sklearn.metrics import accuracy_score, f1_score
import numpy as np

val_probabilities = model.predict(X_val_w2v, verbose=0)
val_predictions = np.argmax(val_probabilities, axis=1)

print(
    "Validation accuracy:",
    round(accuracy_score(y_val_encoded, val_predictions), 4)
)

print(
    "Validation macro F1:",
    round(f1_score(
        y_val_encoded,
        val_predictions,
        average="macro"
    ), 4)
)

Validation accuracy: 0.8195
Validation macro F1: 0.8185


In [23]:
from sklearn.linear_model import LogisticRegression

baseline_model = LogisticRegression(max_iter=1000)
baseline_model.fit(X_train_tfidf, y_train)

baseline_predictions = baseline_model.predict(X_val_tfidf)

print(
    "Baseline validation accuracy:",
    round(accuracy_score(y_val, baseline_predictions), 4)
)

print(
    "Baseline validation macro F1:",
    round(f1_score(
        y_val,
        baseline_predictions,
        average="macro"
    ), 4)
)

Baseline validation accuracy: 0.841
Baseline validation macro F1: 0.834


In [24]:
import pandas as pd

# Convert predicted category IDs back to names
nn_predicted_labels = label_encoder.inverse_transform(val_predictions)

results = pd.DataFrame({
    "message": X_val.to_numpy(),
    "actual_category": y_val.to_numpy(),
    "predicted_category": nn_predicted_labels
})

mistakes = results[
    results["actual_category"] != results["predicted_category"]
]

print("Incorrect predictions:", len(mistakes))
display(mistakes.head(10))

Incorrect predictions: 361


,message,actual_category,predicted_category
0,I put some money in my account the other day a...,balance_not_updated_after_cheque_or_cash_deposit,topping_up_by_card
1,I do not think the pop up went through yet.,pending_top_up,pending_card_payment
7,I withdrew money from the ATM and was overcha...,wrong_exchange_rate_for_cash_withdrawal,cash_withdrawal_charge
10,Where do I find PIN for my card?,pin_blocked,get_physical_card
13,Do i have to wait to get the pin or can i set ...,get_physical_card,change_pin
24,Are you aware of any charges for receiving money?,top_up_by_bank_transfer_charge,transfer_fee_charged
28,Can I use my card in Austria?,country_support,card_acceptance
37,I believe you charged me too much to exchange ...,card_payment_wrong_exchange_rate,wrong_exchange_rate_for_cash_withdrawal
40,Why am I being charged again?,transaction_charged_twice,card_payment_fee_charged
44,I can't access my account.,passcode_forgotten,lost_or_stolen_phone


In [25]:
from sklearn.metrics import classification_report

report = classification_report(
    y_val_encoded,
    val_predictions,
    labels=np.arange(len(label_encoder.classes_)),
    target_names=label_encoder.classes_,
    output_dict=True,
    zero_division=0
)

category_report = pd.DataFrame(report).T.loc[
    label_encoder.classes_,
    ["precision", "recall", "f1-score", "support"]
]

display(category_report.sort_values("f1-score").head(10))

,precision,recall,f1-score,support
topping_up_by_card,0.625000,0.476190,0.540541,21.0
order_physical_card,0.652174,0.625000,0.638298,24.0
transfer_not_received_by_recipient,0.700000,0.617647,0.656250,34.0
virtual_card_not_working,0.714286,0.625000,0.666667,8.0
get_physical_card,0.681818,0.714286,0.697674,21.0
supported_cards_and_currencies,0.720000,0.692308,0.705882,26.0
balance_not_updated_after_bank_transfer,0.741935,0.676471,0.707692,34.0
top_up_reverted,0.638889,0.793103,0.707692,29.0
card_payment_not_recognised,0.785714,0.647059,0.709677,34.0
card_not_working,0.750000,0.681818,0.714286,22.0


In [26]:
# Actual topping_up_by_card messages that the model got wrong
top_up_mistakes = mistakes[
    mistakes["actual_category"] == "topping_up_by_card"
]

# Show full messages without cutting the text
with pd.option_context("display.max_colwidth", None):
    display(top_up_mistakes.head(5))

,message,actual_category,predicted_category
163,can other people use the app to add money to my account,topping_up_by_card,supported_cards_and_currencies
577,Can you tell me how to top up on my card?,topping_up_by_card,top_up_limits
636,what happened to the money after i put in the wrong info and it got declined,topping_up_by_card,declined_cash_withdrawal
669,How do i set up my credit card to send transfers?,topping_up_by_card,top_up_by_card_charge
926,Can a friend top up my account,topping_up_by_card,top_up_failed


In [27]:
comparison = results.copy()

# Predictions are in the same validation-message order
comparison["baseline_prediction"] = baseline_predictions

# Messages from this category that the neural network got wrong
selected = comparison[
    (comparison["actual_category"] == "topping_up_by_card")
    & (comparison["predicted_category"] != comparison["actual_category"])
]

with pd.option_context("display.max_colwidth", None):
    display(selected.head(5))

,message,actual_category,predicted_category,baseline_prediction
163,can other people use the app to add money to my account,topping_up_by_card,supported_cards_and_currencies,supported_cards_and_currencies
577,Can you tell me how to top up on my card?,topping_up_by_card,top_up_limits,top_up_failed
636,what happened to the money after i put in the wrong info and it got declined,topping_up_by_card,declined_cash_withdrawal,declined_transfer
669,How do i set up my credit card to send transfers?,topping_up_by_card,top_up_by_card_charge,topping_up_by_card
926,Can a friend top up my account,topping_up_by_card,top_up_failed,topping_up_by_card


In [28]:
baseline_report = classification_report(
    y_val,
    baseline_predictions,
    labels=label_encoder.classes_,
    output_dict=True,
    zero_division=0
)

baseline_category_report = pd.DataFrame(baseline_report).T

f1_comparison = pd.DataFrame({
    "Neural_network_F1": category_report["f1-score"],
    "Baseline_F1": baseline_category_report.loc[
        label_encoder.classes_, "f1-score"
    ]
})

display(
    f1_comparison.loc[
        category_report.sort_values("f1-score").head(10).index
    ].round(3)
)

,Neural_network_F1,Baseline_F1
topping_up_by_card,0.541,0.703
order_physical_card,0.638,0.696
transfer_not_received_by_recipient,0.656,0.857
virtual_card_not_working,0.667,0.222
get_physical_card,0.698,0.870
supported_cards_and_currencies,0.706,0.808
balance_not_updated_after_bank_transfer,0.708,0.720
top_up_reverted,0.708,0.842
card_payment_not_recognised,0.710,0.667
card_not_working,0.714,0.780


In [29]:
# Fresh predictions from the current neural network
val_predictions = model.predict(
    X_val_w2v, verbose=0
).argmax(axis=1)

print("Current NN accuracy:",
      round(accuracy_score(y_val_encoded, val_predictions), 4))

print("Current NN macro F1:",
      round(f1_score(
          y_val_encoded, val_predictions, average="macro"
      ), 4))

# Fresh category report
report = classification_report(
    y_val_encoded,
    val_predictions,
    labels=np.arange(len(label_encoder.classes_)),
    target_names=label_encoder.classes_,
    output_dict=True,
    zero_division=0
)

category_report = pd.DataFrame(report).T.loc[
    label_encoder.classes_,
    ["precision", "recall", "f1-score", "support"]
]

display(category_report.loc[["topping_up_by_card"]])

Current NN accuracy: 0.8195
Current NN macro F1: 0.8185


,precision,recall,f1-score,support
topping_up_by_card,0.625,0.47619,0.540541,21.0


In [30]:
baseline_predictions = baseline_model.predict(X_val_tfidf)

print("Baseline accuracy:",
      round(accuracy_score(y_val, baseline_predictions), 4))

print("Baseline macro F1:",
      round(f1_score(
          y_val,
          baseline_predictions,
          average="macro"
      ), 4))

Baseline accuracy: 0.841
Baseline macro F1: 0.834


In [31]:
# Baseline: use the vocabulary learned from training
X_test_tfidf = vectorizer.transform(test_data["text"])

# Neural network: use the same tokenizer and learned Word2Vec
test_tokens = test_data["text"].apply(get_tokens)
X_test_w2v = np.vstack(test_tokens.apply(get_message_vector))

# Use the same category-to-ID mapping
y_test_encoded = label_encoder.transform(test_data["category"])

print("Test TF-IDF shape:", X_test_tfidf.shape)
print("Test Word2Vec shape:", X_test_w2v.shape)
print("Word2Vec has NaN:", np.isnan(X_test_w2v).any())

Test TF-IDF shape: (3080, 20915)
Test Word2Vec shape: (3080, 100)
Word2Vec has NaN: False


In [32]:
# Baseline predictions: category names
test_baseline_predictions = baseline_model.predict(X_test_tfidf)

# Neural network predictions: category IDs
test_nn_predictions = model.predict(
    X_test_w2v, verbose=0
).argmax(axis=1)

test_results = pd.DataFrame({
    "Model": [
        "TF-IDF + Logistic Regression",
        "Word2Vec + Neural Network"
    ],
    "Test accuracy": [
        accuracy_score(
            test_data["category"], test_baseline_predictions
        ),
        accuracy_score(
            y_test_encoded, test_nn_predictions
        )
    ],
    "Test macro F1": [
        f1_score(
            test_data["category"],
            test_baseline_predictions,
            average="macro"
        ),
        f1_score(
            y_test_encoded,
            test_nn_predictions,
            average="macro"
        )
    ]
})

display(test_results.round(4))

,Model,Test accuracy,Test macro F1
0,TF-IDF + Logistic Regression,0.8406,0.8378
1,Word2Vec + Neural Network,0.8221,0.8215


In [33]:
def predict_category(message):
    if not isinstance(message, str) or not message.strip():
        return "Please enter a banking message."

    # Apply the TF-IDF transformation learned during training
    message_vector = vectorizer.transform([message])

    # Predict the category
    return baseline_model.predict(message_vector)[0]


message = "My card has not arrived yet."

print("Message:", message)
print("Predicted category:", predict_category(message))

Message: My card has not arrived yet.
Predicted category: transfer_not_received_by_recipient


In [34]:
message = "My card has not arrived yet."
message_vector = vectorizer.transform([message])

print("Vectorizer features:", message_vector.shape[1])
print("Model expected features:", baseline_model.n_features_in_)
print("Known features in message:", message_vector.nnz)

# Show the model's top 3 predictions
probabilities = baseline_model.predict_proba(message_vector)[0]
top_indices = np.argsort(probabilities)[-3:][::-1]

for index in top_indices:
    print(
        baseline_model.classes_[index],
        "→",
        round(float(probabilities[index]), 4)
    )

Vectorizer features: 20915
Model expected features: 20915
Known features in message: 11
transfer_not_received_by_recipient → 0.1067
card_arrival → 0.1043
lost_or_stolen_card → 0.038


In [35]:
tokens = get_tokens(message)
message_w2v = get_message_vector(tokens).reshape(1, -1)

nn_probabilities = model.predict(message_w2v, verbose=0)[0]
top_indices = np.argsort(nn_probabilities)[-3:][::-1]

for index in top_indices:
    print(
        label_encoder.inverse_transform([index])[0],
        "→",
        round(float(nn_probabilities[index]), 4)
    )

card_arrival → 0.9988
card_delivery_estimate → 0.0006
transfer_not_received_by_recipient → 0.0002


In [36]:
required = [
    "model",
    "baseline_model",
    "vectorizer",
    "w2v_model",
    "label_encoder"
]

for name in required:
    status = "Available" if name in globals() else "Missing"
    print(f"{name}: {status}")

model: Available
baseline_model: Available
vectorizer: Available
w2v_model: Available
label_encoder: Available


In [37]:
from pathlib import Path
import joblib

save_dir = Path("/content/banking_models")
save_dir.mkdir(exist_ok=True)

# Neural network
model.save(save_dir / "neural_network.keras")

# Word2Vec vectors
w2v_model.save(str(save_dir / "word2vec.model"))

# Baseline and preprocessing objects
joblib.dump(baseline_model, save_dir / "baseline_model.joblib")
joblib.dump(vectorizer, save_dir / "tfidf_vectorizer.joblib")
joblib.dump(label_encoder, save_dir / "label_encoder.joblib")

print("Saved files:")
for file in sorted(save_dir.iterdir()):
    print(file.name)

Saved files:
baseline_model.joblib
label_encoder.joblib
neural_network.keras
tfidf_vectorizer.joblib
word2vec.model


In [38]:
import tensorflow as tf
import numpy as np
from gensim.models import Word2Vec

# Reload with separate names
loaded_nn = tf.keras.models.load_model(
    save_dir / "neural_network.keras"
)
loaded_baseline = joblib.load(save_dir / "baseline_model.joblib")
loaded_vectorizer = joblib.load(save_dir / "tfidf_vectorizer.joblib")
loaded_encoder = joblib.load(save_dir / "label_encoder.joblib")
loaded_w2v = Word2Vec.load(str(save_dir / "word2vec.model"))

message = "My card has not arrived yet."

# Baseline: compare original and reloaded predictions
original_baseline = baseline_model.predict(
    vectorizer.transform([message])
)[0]

reloaded_baseline = loaded_baseline.predict(
    loaded_vectorizer.transform([message])
)[0]

# Neural network: prepare input using reloaded Word2Vec
tokens = get_tokens(message)
vectors = [
    loaded_w2v.wv[word]
    for word in tokens
    if word in loaded_w2v.wv
]

message_vector = (
    np.mean(vectors, axis=0)
    if vectors
    else np.zeros(loaded_w2v.vector_size, dtype=np.float32)
).reshape(1, -1)

original_probs = model.predict(
    get_message_vector(tokens).reshape(1, -1), verbose=0
)[0]

reloaded_probs = loaded_nn.predict(message_vector, verbose=0)[0]

print("Baseline predictions match:",
      original_baseline == reloaded_baseline)

print("Neural network probabilities match:",
      np.allclose(original_probs, reloaded_probs, atol=1e-6))

print("Reloaded NN category:",
      loaded_encoder.inverse_transform(
          [int(np.argmax(reloaded_probs))]
      )[0])

Baseline predictions match: True
Neural network probabilities match: True
Reloaded NN category: card_arrival


In [39]:
import shutil
from google.colab import files

zip_path = shutil.make_archive(
    "/content/banking_models",
    "zip",
    root_dir="/content",
    base_dir="banking_models"
)

files.download(zip_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [42]:
%pip -q install gradio

In [43]:
import gradio as gr

def app_predict(message):
    if not message or not message.strip():
        return "Please enter a banking message.", ""

    # Baseline
    text_features = loaded_vectorizer.transform([message])
    baseline_category = loaded_baseline.predict(text_features)[0]

    # Neural network
    tokens = get_tokens(message)
    vectors = [
        loaded_w2v.wv[word]
        for word in tokens
        if word in loaded_w2v.wv
    ]

    if not vectors:
        return baseline_category, "No known words found."

    message_vector = np.mean(vectors, axis=0).reshape(1, -1)
    probabilities = loaded_nn.predict(message_vector, verbose=0)[0]

    nn_category = loaded_encoder.inverse_transform(
        [int(np.argmax(probabilities))]
    )[0]

    return baseline_category, nn_category


demo = gr.Interface(
    fn=app_predict,
    inputs=gr.Textbox(
        label="Customer banking message",
        placeholder="Example: My card has not arrived yet.",
        lines=3
    ),
    outputs=[
        gr.Textbox(label="TF-IDF + Logistic Regression"),
        gr.Textbox(label="Word2Vec + Neural Network")
    ],
    title="Banking Query Classifier",
    description=(
        "Compare two models across 77 banking query categories. "
        "Predictions may be incorrect; this is a portfolio demo."
    ),
    examples=[
        ["My card has not arrived yet."],
        ["I was charged twice for one payment."],
        ["How do I change my PIN?"]
    ]
)

demo.launch(share=False)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>

In [44]:
cash_examples = data_clean[
    data_clean["text"].str.contains(
        r"no cash|no money|didn't receive|did not receive|dispense",
        case=False,
        na=False
    )
    & data_clean["category"].str.contains(
        "cash|withdrawal", case=False, na=False
    )
]

with pd.option_context("display.max_colwidth", None):
    display(cash_examples[["text", "category"]].head(10))

,text,category
751,"When I used one of the ATMs in the city centre today, I tried to take out some money, but for whatever reason, the machine declined my card. Just checked it out in my account and that amount is still showing as pending! Is there any way that you can cancel this transaction, because I did not receive any money whatsoever from the ATM",pending_cash_withdrawal
753,"Hello. I looked at my account and see that a transaction I tried to do is pending, but I did not complete that transaction. When I was at the ATM, the machine declined my card. Can you make sure that transaction does not go through because I definitely did not receive the money.",pending_cash_withdrawal
793,I tried to get some money but the machine was not working .The transaction still seems in progress! Can you please check what's going on.I don't want to be charged for money that I did not received.,pending_cash_withdrawal
799,I didn't receive my money earlier and it says the transaction is still in progress. Can you fix it?,pending_cash_withdrawal
800,"Hello. I have a transaction in my account listed as pending. I tried to use an ATM at the city centre, but the machine declined my card, and I did not receive the money. Why is the transaction still pending if I did not receive the money.",pending_cash_withdrawal
851,Please cancel the latest ATM charge on my card as my card was declined earlier today and I did not receive that cash.,pending_cash_withdrawal
874,"My card was declined and I did not receive money from the ATM earlier, yet it show on my account that it is pending.",pending_cash_withdrawal
2364,I didn't receive the amount of cash I requested for the ATM.,wrong_amount_of_cash_received
2391,"Hello, I didn't receive the right amount on my ATM transaction, the app shows the correct amount how can we fix this",wrong_amount_of_cash_received
2392,I did not receive my full withdrawal amount,wrong_amount_of_cash_received


In [45]:
%%writefile /content/app.py
from pathlib import Path

import contractions
import gradio as gr
import joblib
import numpy as np
import tensorflow as tf
from gensim.models import Word2Vec
from gensim.utils import tokenize

# Models folder should be beside app.py
MODEL_DIR = Path(__file__).resolve().parent / "banking_models"

baseline = joblib.load(MODEL_DIR / "baseline_model.joblib")
vectorizer = joblib.load(MODEL_DIR / "tfidf_vectorizer.joblib")
encoder = joblib.load(MODEL_DIR / "label_encoder.joblib")

nn_model = tf.keras.models.load_model(
    MODEL_DIR / "neural_network.keras",
    compile=False
)
w2v = Word2Vec.load(str(MODEL_DIR / "word2vec.model"))


def get_tokens(message):
    text = message.lower().replace("’", "'")
    return list(tokenize(contractions.fix(text)))


def app_predict(message):
    if not message or not message.strip():
        return "Please enter a banking message.", ""

    baseline_category = baseline.predict(
        vectorizer.transform([message])
    )[0]

    vectors = [
        w2v.wv[word]
        for word in get_tokens(message)
        if word in w2v.wv
    ]

    if not vectors:
        return baseline_category, "No known words found."

    message_vector = np.mean(vectors, axis=0).reshape(1, -1)
    probabilities = nn_model.predict(
        message_vector, verbose=0
    )[0]

    nn_category = encoder.inverse_transform(
        [int(np.argmax(probabilities))]
    )[0]

    return baseline_category, nn_category


demo = gr.Interface(
    fn=app_predict,
    inputs=gr.Textbox(
        label="Customer banking message",
        placeholder="Example: My card has not arrived yet.",
        lines=3
    ),
    outputs=[
        gr.Textbox(label="TF-IDF + Logistic Regression"),
        gr.Textbox(label="Word2Vec + Neural Network")
    ],
    title="Banking Query Classifier",
    description=(
        "Compare two models across 77 banking query categories. "
        "Predictions may be incorrect; this is a portfolio demo."
    ),
    examples=[
        ["My card has not arrived yet."],
        ["I was charged twice for one payment."],
        ["How do I change my PIN?"]
    ]
)

if __name__ == "__main__":
    demo.launch(share=False)

Writing /content/app.py


In [46]:
from importlib.metadata import version
from pathlib import Path
import platform

packages = [
    "contractions",
    "gradio",
    "joblib",
    "numpy",
    "scipy",
    "scikit-learn",
    "tensorflow",
    "gensim"
]

requirements = "\n".join(
    f"{package}=={version(package)}"
    for package in packages
)

Path("/content/requirements.txt").write_text(
    requirements + "\n",
    encoding="utf-8"
)

print("Python version:", platform.python_version())
print("\nSaved requirements:")
print(requirements)

Python version: 3.13.16

Saved requirements:
contractions==0.1.73
gradio==6.29.0
joblib==1.6.0
numpy==2.1.3
scipy==1.16.3
scikit-learn==1.6.1
tensorflow==2.21.0
gensim==4.4.0


In [47]:
import subprocess
import sys

check_code = """
import app

message = "My card has not arrived yet."
baseline, neural_network = app.app_predict(message)

print("Baseline:", baseline)
print("Neural network:", neural_network)
"""

result = subprocess.run(
    [sys.executable, "-c", check_code],
    cwd="/content",
    capture_output=True,
    text=True
)

print("Exit code:", result.returncode)
print(result.stdout)

if result.returncode != 0:
    print("Error details:")
    print(result.stderr)

Exit code: 0
Baseline: transfer_not_received_by_recipient
Neural network: card_arrival



In [48]:
%%writefile /content/README.md
# Banking Query Classification

A portfolio NLP project that classifies English customer banking
messages into 77 categories using the BANKING77 dataset.

The Gradio app compares:
- TF-IDF + Logistic Regression
- Averaged Word2Vec embeddings + a dense neural network

## Business objective

Support customer-query routing by predicting the banking category
of an incoming message. The demo predicts a category; it does not
resolve complaints or perform banking transactions.

## Dataset and preparation

Source: https://github.com/PolyAI-LDN/task-specific-datasets/tree/master/banking_data

- Original training messages: 10,003
- Original test messages: 3,080
- No missing values, exact duplicate rows, or blank messages found.
- No conflicting training labels found after normalizing case
  and whitespace for comparison.
- Seven normalized messages appeared in both train and test.
  Their training rows were removed; the test set was preserved.
- Cleaned training data: 9,996 messages.
- Stratified split: 7,996 training and 2,000 validation messages.

TF-IDF and Word2Vec were fitted on training messages only.
Category encoding was also fitted on training labels only.

## Methods

### Baseline

TF-IDF with single words and word pairs, followed by
Logistic Regression.

### Neural network

Contraction expansion and tokenization, followed by Word2Vec:
100 dimensions, Skip-gram, window 5, minimum count 1, 20 epochs.

Known word vectors are averaged to produce one message vector.
The classifier has hidden layers of 128 and 64 ReLU neurons,
dropout, and 77 softmax outputs.

## Recorded results

| Model | Validation accuracy | Validation macro F1 | Test accuracy | Test macro F1 |
|---|---:|---:|---:|---:|
| TF-IDF + Logistic Regression | 0.8410 | 0.8340 | 0.8406 | 0.8378 |
| Word2Vec + Neural Network | 0.8195 | 0.8185 | 0.8221 | 0.8215 |

Results reflect the final evaluated notebook models.
The baseline performed better overall, while the neural network
correctly handled some messages that the baseline missed.

## Limitations

- Related banking intents can be confused.
- Averaging word vectors discards word order.
- Unseen Word2Vec words are skipped.
- Every supported input receives one of the known categories;
  out-of-domain detection is not implemented.
- Probability scores are not calibrated confidence guarantees.
- Neural-network results can vary across training runs.
- This is a portfolio demo, not a production banking system.

## Project files

- `app.py`: standalone Gradio app and inference preprocessing.
- `requirements.txt`: direct dependency versions from Colab.
- `banking_models/`: saved models, vectorizer, and label encoder.
- Training notebook: add the exported Colab `.ipynb` file.

## Run locally

The app was verified in the existing Colab environment using
Python 3.13.16. A fresh dependency installation has not yet been tested.

Keep `app.py` and `banking_models/` in the same project folder.

```bash
python -m venv .venv
```

Activate the environment:

Windows:
```bash
.venv\Scripts\activate
```

macOS/Linux:
```bash
source .venv/bin/activate
```

Then run:
```bash
python -m pip install -r requirements.txt
python app.py
```

Open the local URL printed in the terminal.

Only load model files from a trusted source.

## Verification completed

- Saved and reloaded model predictions matched.
- Standalone inference succeeded in a separate Python process.
- The Gradio interface returned predictions for entered messages.

## Future improvements

Compare contextual text embeddings, investigate confusing intents,
and validate a human-review mechanism for uncertain predictions.

Writing /content/README.md


In [49]:
from pathlib import Path
import shutil
from google.colab import files

project_dir = Path("/content/banking_query_classifier")
project_dir.mkdir(exist_ok=True)

# Copy app and documentation
for name in ["app.py", "requirements.txt", "README.md"]:
    shutil.copy2(Path("/content") / name, project_dir / name)

# Copy all model files, including any Word2Vec supporting files
shutil.copytree(
    "/content/banking_models",
    project_dir / "banking_models",
    dirs_exist_ok=True
)

zip_path = shutil.make_archive(
    "/content/banking_query_classifier",
    "zip",
    root_dir="/content",
    base_dir="banking_query_classifier"
)

files.download(zip_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>